# 🎯 Prompt Optimization

**Systematically improve prompts for better results**

---

## 📋 Overview

**What you'll learn:**
- A/B testing prompts
- Metrics for evaluation
- Iterative improvement strategies
- Prompt versioning
- Automated optimization

**Time estimate:** ⏱️ 60 minutes | **Difficulty:** 🔴 Advanced

---

In [ ]:
from openai import OpenAI
import os
from typing import List, Dict, Callable
import json
from datetime import datetime
import pandas as pd

client = OpenAI(api_key=os.getenv('OPENAI_API_KEY'))

print("✅ Setup complete")

## 🎯 Why Optimize Prompts?

**Small prompt changes = Big accuracy gains**

### Example:
```python
# ❌ Basic prompt (70% accuracy)
"Classify the sentiment"

# ✅ Optimized prompt (90% accuracy)
"Classify the sentiment as Positive, Negative, or Neutral.
Only respond with one word."
```

### What to Optimize:
- 📝 **Wording**: Clear vs vague instructions
- 🎨 **Format**: Structure and layout
- 📚 **Examples**: Number and quality
- 🎛️ **Parameters**: Temperature, max_tokens
- 🔄 **Order**: Instruction placement

## 📊 Evaluation Metrics

In [ ]:
class PromptEvaluator:
    """Evaluate prompt performance."""
    
    def __init__(self, test_cases: List[Dict]):
        """
        Args:
            test_cases: List of {'input': ..., 'expected': ...}
        """
        self.test_cases = test_cases
    
    def evaluate_prompt(
        self,
        prompt_template: str,
        model: str = "gpt-3.5-turbo",
        temperature: float = 0
    ) -> Dict:
        """Evaluate a prompt on test cases."""
        
        results = []
        correct = 0
        total_tokens = 0
        
        for test in self.test_cases:
            # Format prompt
            prompt = prompt_template.format(input=test['input'])
            
            # Get prediction
            response = client.chat.completions.create(
                model=model,
                messages=[{"role": "user", "content": prompt}],
                temperature=temperature,
                max_tokens=50
            )
            
            prediction = response.choices[0].message.content.strip()
            expected = test['expected']
            
            # Check if correct
            is_correct = prediction.lower() == expected.lower()
            if is_correct:
                correct += 1
            
            total_tokens += response.usage.total_tokens
            
            results.append({
                'input': test['input'],
                'expected': expected,
                'predicted': prediction,
                'correct': is_correct
            })
        
        accuracy = correct / len(self.test_cases)
        avg_tokens = total_tokens / len(self.test_cases)
        
        return {
            'accuracy': accuracy,
            'correct': correct,
            'total': len(self.test_cases),
            'avg_tokens': avg_tokens,
            'total_tokens': total_tokens,
            'results': results
        }

# Create test cases
test_cases = [
    {"input": "I love this product!", "expected": "Positive"},
    {"input": "This is terrible.", "expected": "Negative"},
    {"input": "It's okay.", "expected": "Neutral"},
    {"input": "Best purchase ever!", "expected": "Positive"},
    {"input": "Waste of money.", "expected": "Negative"},
    {"input": "Works as expected.", "expected": "Neutral"},
]

evaluator = PromptEvaluator(test_cases)

# Test basic prompt
basic_prompt = "Classify the sentiment: {input}"

results = evaluator.evaluate_prompt(basic_prompt)

print(f"📊 Basic Prompt Results:")
print(f"  Accuracy: {results['accuracy']*100:.1f}%")
print(f"  Correct: {results['correct']}/{results['total']}")
print(f"  Avg tokens: {results['avg_tokens']:.0f}")

## 🔬 A/B Testing Prompts

In [ ]:
# Test multiple prompt variations
prompt_variations = {
    "v1_basic": "Classify the sentiment: {input}",
    
    "v2_specific": """Classify the sentiment as Positive, Negative, or Neutral.
Text: {input}
Sentiment:""",
    
    "v3_constrained": """Classify the sentiment as Positive, Negative, or Neutral.
Respond with ONLY one word.

Text: {input}
Sentiment:""",
    
    "v4_role": """You are a sentiment analysis expert.
Classify the sentiment as Positive, Negative, or Neutral.
Respond with only the classification.

Text: {input}
Sentiment:""",
}

print("🧪 A/B Testing Prompt Variations\n")
print("="*60)

comparison = []

for version, prompt in prompt_variations.items():
    print(f"\n Testing {version}...")
    results = evaluator.evaluate_prompt(prompt)
    
    comparison.append({
        'version': version,
        'accuracy': results['accuracy'],
        'avg_tokens': results['avg_tokens']
    })
    
    print(f"  Accuracy: {results['accuracy']*100:.1f}%")
    print(f"  Avg tokens: {results['avg_tokens']:.0f}")

# Summary
print("\n" + "="*60)
print("\n📊 Comparison Summary:\n")

df = pd.DataFrame(comparison).sort_values('accuracy', ascending=False)
print(df.to_string(index=False))

best = df.iloc[0]
print(f"\n✅ Best prompt: {best['version']} ({best['accuracy']*100:.1f}% accuracy)")

## 🔄 Iterative Optimization

In [ ]:
class PromptOptimizer:
    """Iteratively optimize prompts."""
    
    def __init__(self, evaluator: PromptEvaluator):
        self.evaluator = evaluator
        self.history = []
    
    def optimize(
        self,
        base_prompt: str,
        modifications: List[Callable],
        max_iterations: int = 5
    ) -> Dict:
        """Optimize prompt through iterations."""
        
        current_prompt = base_prompt
        best_prompt = base_prompt
        best_score = 0
        
        print(f"🔄 Starting optimization (max {max_iterations} iterations)\n")
        
        for iteration in range(max_iterations):
            print(f"Iteration {iteration + 1}:")
            
            # Evaluate current prompt
            results = self.evaluator.evaluate_prompt(current_prompt)
            score = results['accuracy']
            
            print(f"  Current accuracy: {score*100:.1f}%")
            
            # Update best if better
            if score > best_score:
                best_score = score
                best_prompt = current_prompt
                print(f"  ✅ New best!")
            
            # Record history
            self.history.append({
                'iteration': iteration + 1,
                'prompt': current_prompt,
                'accuracy': score,
                'tokens': results['avg_tokens']
            })
            
            # Try modifications
            candidates = []
            for mod_func in modifications:
                modified = mod_func(current_prompt)
                if modified != current_prompt:
                    mod_results = self.evaluator.evaluate_prompt(modified)
                    candidates.append({
                        'prompt': modified,
                        'score': mod_results['accuracy']
                    })
            
            # Pick best candidate
            if candidates:
                best_candidate = max(candidates, key=lambda x: x['score'])
                if best_candidate['score'] > score:
                    current_prompt = best_candidate['prompt']
                    print(f"  📈 Improved to {best_candidate['score']*100:.1f}%")
                else:
                    print(f"  📊 No improvement found")
                    break
            else:
                print(f"  🛑 No more modifications to try")
                break
            
            print()
        
        return {
            'best_prompt': best_prompt,
            'best_score': best_score,
            'iterations': len(self.history),
            'history': self.history
        }

# Define modification functions
def add_role(prompt: str) -> str:
    """Add role instruction."""
    if "You are" not in prompt:
        return f"You are a sentiment analysis expert.\n{prompt}"
    return prompt

def add_constraint(prompt: str) -> str:
    """Add output constraint."""
    if "ONLY" not in prompt:
        return f"{prompt}\nRespond with ONLY one word."
    return prompt

def add_examples(prompt: str) -> str:
    """Add few-shot examples."""
    if "Example" not in prompt:
        examples = """\nExamples:
- "Great!" → Positive
- "Terrible" → Negative
- "Okay" → Neutral
"""
        return examples + prompt
    return prompt

# Run optimization
optimizer = PromptOptimizer(evaluator)

result = optimizer.optimize(
    base_prompt="Classify sentiment: {input}",
    modifications=[add_role, add_constraint, add_examples],
    max_iterations=5
)

print("="*60)
print(f"\n🏆 Optimization Complete!")
print(f"  Best accuracy: {result['best_score']*100:.1f}%")
print(f"  Iterations: {result['iterations']}")
print(f"\n  Best prompt:\n{result['best_prompt'][:100]}...")

## 📦 Prompt Versioning

In [ ]:
from pathlib import Path
from dataclasses import dataclass, asdict

@dataclass
class PromptVersion:
    """Versioned prompt with metadata."""
    version: str
    prompt: str
    description: str
    accuracy: float = 0.0
    created_at: str = None
    tags: List[str] = None
    
    def __post_init__(self):
        if self.created_at is None:
            self.created_at = datetime.now().isoformat()
        if self.tags is None:
            self.tags = []

class PromptRegistry:
    """Manage versioned prompts."""
    
    def __init__(self, registry_dir: str = "prompt_registry"):
        self.registry_dir = Path(registry_dir)
        self.registry_dir.mkdir(exist_ok=True)
        self.prompts = {}
    
    def save(self, prompt_version: PromptVersion):
        """Save prompt version."""
        file_path = self.registry_dir / f"{prompt_version.version}.json"
        
        with open(file_path, 'w') as f:
            json.dump(asdict(prompt_version), f, indent=2)
        
        self.prompts[prompt_version.version] = prompt_version
        print(f"✅ Saved {prompt_version.version}")
    
    def load(self, version: str) -> PromptVersion:
        """Load prompt version."""
        file_path = self.registry_dir / f"{version}.json"
        
        with open(file_path, 'r') as f:
            data = json.load(f)
        
        return PromptVersion(**data)
    
    def list_versions(self) -> List[str]:
        """List all versions."""
        return [f.stem for f in self.registry_dir.glob('*.json')]
    
    def get_best(self, task: str = None) -> PromptVersion:
        """Get best performing prompt."""
        versions = []
        for version_file in self.registry_dir.glob('*.json'):
            with open(version_file, 'r') as f:
                data = json.load(f)
                if task is None or task in data.get('tags', []):
                    versions.append(PromptVersion(**data))
        
        if not versions:
            return None
        
        return max(versions, key=lambda v: v.accuracy)

# Create registry
registry = PromptRegistry()

# Save different versions
versions = [
    PromptVersion(
        version="v1.0.0",
        prompt="Classify sentiment: {input}",
        description="Basic sentiment classifier",
        accuracy=0.70,
        tags=["sentiment", "basic"]
    ),
    PromptVersion(
        version="v1.1.0",
        prompt="""Classify the sentiment as Positive, Negative, or Neutral.
Text: {input}
Sentiment:""",
        description="Added specific output format",
        accuracy=0.85,
        tags=["sentiment", "improved"]
    ),
    PromptVersion(
        version="v2.0.0",
        prompt="""You are a sentiment analysis expert.
Classify the sentiment as Positive, Negative, or Neutral.
Respond with ONLY one word.

Text: {input}
Sentiment:""",
        description="Added role and constraints",
        accuracy=0.92,
        tags=["sentiment", "production"]
    ),
]

for v in versions:
    registry.save(v)

# Get best version
best = registry.get_best(task="sentiment")
print(f"\n🏆 Best prompt: {best.version} ({best.accuracy*100:.1f}% accuracy)")
print(f"   Description: {best.description}")

## 🎛️ Parameter Optimization

In [ ]:
def optimize_temperature(prompt: str, test_cases: List[Dict]) -> Dict:
    """Find optimal temperature."""
    
    temperatures = [0.0, 0.3, 0.5, 0.7, 1.0]
    results = []
    
    print("🌡️ Testing different temperatures...\n")
    
    for temp in temperatures:
        evaluator = PromptEvaluator(test_cases)
        eval_results = evaluator.evaluate_prompt(
            prompt,
            temperature=temp
        )
        
        results.append({
            'temperature': temp,
            'accuracy': eval_results['accuracy'],
            'avg_tokens': eval_results['avg_tokens']
        })
        
        print(f"  temp={temp:.1f} → {eval_results['accuracy']*100:.1f}% accuracy")
    
    # Find best
    best = max(results, key=lambda x: x['accuracy'])
    
    print(f"\n✅ Best temperature: {best['temperature']} ({best['accuracy']*100:.1f}%)")
    
    return results

# Test temperature optimization
prompt = """Classify the sentiment as Positive, Negative, or Neutral.
Respond with ONLY one word.

Text: {input}
Sentiment:"""

temp_results = optimize_temperature(prompt, test_cases)

## ✅ Summary

### Key Optimization Strategies:

1. **🧪 A/B Testing**
   - Test multiple variations
   - Measure on same test set
   - Pick best performer

2. **🔄 Iterative Improvement**
   - Start simple
   - Add one change at a time
   - Measure impact

3. **📦 Version Control**
   - Track all versions
   - Document changes
   - Can rollback if needed

4. **🎛️ Parameter Tuning**
   - Temperature (0-1)
   - Max tokens
   - Stop sequences

### Optimization Checklist:

```python
# 1. Create test set
test_cases = [{"input": ..., "expected": ...}, ...]

# 2. Baseline
v1_accuracy = evaluate(simple_prompt)

# 3. Try improvements
improvements = [
    "Add role",
    "Add constraints",
    "Add examples",
    "Restructure format"
]

# 4. Test each
for improvement in improvements:
    new_accuracy = evaluate(modified_prompt)
    if new_accuracy > best:
        keep_it()

# 5. Version and deploy
save_version("v2.0.0", best_prompt)
```

### Common Improvements:

| Change | Impact | Example |
|--------|--------|----------|
| Add role | +5-10% | "You are an expert..." |
| Constrain output | +10-15% | "Respond with ONLY..." |
| Add examples | +15-20% | Few-shot learning |
| Better structure | +5-10% | Clear sections |
| Optimize temp | +5% | Test 0, 0.3, 0.7 |

### Metrics to Track:

1. **Accuracy**: % correct predictions
2. **Cost**: Tokens used per request
3. **Latency**: Response time
4. **Consistency**: Same input → same output

### Prompt Evolution:

```
v1.0: "Classify: {input}"                   → 70%
v1.1: "Classify as Positive/Negative..."    → 80%
v1.2: + "Respond with ONLY one word"        → 85%
v2.0: + "You are an expert..."              → 90%
v2.1: + Few-shot examples                   → 93%
```

### Production Best Practices:

1. **Always test on held-out data**
2. **Track metrics over time**
3. **Version every change**
4. **A/B test in production**
5. **Monitor for drift**

### Tools for Optimization:

- **PromptLayer**: Prompt versioning
- **LangSmith**: Evaluation & testing
- **Weights & Biases**: Experiment tracking
- **Custom**: Build your own (like we did!)

### Next: `04_embeddings_vectors/04_semantic_search.ipynb`